# Placement probe: does single-operation behaviour depend on the slot position?

**Inference only.** This notebook loads the already-selected checkpoints (`runs/<model>/best.pt`) of the
mixed-length experiments (notebooks 04 and 05). It does not train, fine-tune or re-select anything, and it uses no
optimizer. AR models decode free-running.

**Motivation.** In the standard test, a single operation is encoded as `[X, PAD, PAD, PAD, PAD]`. Several models
output "X then C" (A→AC, B→BC, C→CC = identity) on it. This probe asks whether that behaviour depends on where the
operation sits:

* **Singles:** A, B and C, each placed in slot 1, 2, 3, 4 and 5 (all other slots PAD), on the 1000 test base grids.
* **Control:** the experiment's trained length-two pairs in all 10 order-preserving placements (all of which occur
  in training), on 200 grids. This shows whether non-canonical placements are handled correctly for a TRAINED
  length.

Each prediction is checked against the truth and against explicit candidate rules: "X then C", "X twice", identity,
and every original training pair. It is also checked against all 288 legal transformations (the dominant wrong one
is reported). Results are behavioural, from one seed, and non-causal.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION ---------------------------------------------------------------------------------------
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
MIXED_ROOT = os.environ.get("DECOMP_MIXED_ROOT") or f"{RESULTS_ROOT}/mixed_length_02345"
EXPERIMENTS = ["mixed_02345_distinct_pairs", "mixed_02345_all_pairs"]
MODELS = os.environ.get("DECOMP_MODELS", "transformer_ar,looped_transformer_ar,transformer_encoder,"
                        "looped_transformer_encoder,vit,looped_vit,trm,rescnn_film,unet_film").split(",")
N_SINGLE_GRIDS = int(os.environ.get("DECOMP_PROBE_GRIDS", 1000))   # singles: 3 ops x 5 slots x grids
N_PAIR_GRIDS = int(os.environ.get("DECOMP_PROBE_PAIR_GRIDS", 200))  # control: pairs x 10 placements x grids
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
DTYPE = "auto"
OUTPUT_DIR = f"{MIXED_ROOT}/placement_probe"
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("reading checkpoints from:", MIXED_ROOT)
print("writing probe outputs to:", OUTPUT_DIR)


In [ ]:
# --- Run the probe (cached per experiment/model; inference only) -------------------------------------------
import pandas as pd
from IPython.display import Image, display
from decompositionality.placement_probe import run_probe, plot_probe
pd.set_option("display.max_rows", 400); pd.set_option("display.width", 250)
res, out = run_probe(MIXED_ROOT, EXPERIMENTS, MODELS, OUTPUT_DIR, N_SINGLE_GRIDS, N_PAIR_GRIDS, DEVICE, DTYPE)
paths = plot_probe(res, out / "figures")
for p in paths:
    if str(p).endswith(".png"):
        display(Image(filename=str(p)))


In [ ]:
# --- Numeric tables -----------------------------------------------------------------------------------------
cols = ["experiment", "model", "string", "positions", "exact_correct", "n", "exact_acc", "match: X then C",
        "match: X twice", "match: identity", "any_legal_match_rate", "dominant_wrong_transform", "dominant_wrong_share"]
singles = res[res.kind == "single"][cols]
display(singles.style.format(precision=3).hide(axis="index"))
ctrl = res[res.kind == "pair_control"]
display(ctrl.groupby(["experiment", "model", "canonical_placement"])[["exact_correct", "n"]].sum()
        .assign(exact_acc=lambda d: d.exact_correct / d.n))


In [ ]:
# --- Export -------------------------------------------------------------------------------------------------
import shutil
arch = shutil.make_archive(str(out) + "_export", "zip", root_dir=out, base_dir=".")
print("tables/figures:", out, "| archive:", arch)


Outputs: `<RESULTS_ROOT>/mixed_length_02345/placement_probe/` (per-model `probe_predictions.npz` + `probe_meta.json` with the checkpoint step, `placement_probe_results.csv`, `figures/`).